In [ ]:
# import the pkl file top_batch1_sampled_fluxes_timepoint0.pkl and read the samples

import numpy as np
import cobra
from cobra.io import load_model, read_sbml_model
import pandas as pd
import os
import re


In [ ]:
raw_model = 'iCHO2441.xml'
model = read_sbml_model(raw_model)


In [ ]:
old_model = 'iCHOv1_DG44_final.xml'
old_model = read_sbml_model(old_model)
r1_1 = old_model.reactions.get_by_id('CSm')

In [ ]:
model_reactions = set([rxn.id for rxn in model.reactions])
old_model_reactions = set([rxn.id for rxn in old_model.reactions])
common_reactions = model_reactions.intersection(old_model_reactions)
print('Number of common reactions between model and old model is ', len(common_reactions))
for rxn_id in common_reactions:
    rxn_model = model.reactions.get_by_id(rxn_id)
    rxn_old_model = old_model.reactions.get_by_id(rxn_id)
    rxn_model.subsystem = rxn_old_model.subsystem

In [ ]:
tmp = {}
for i in model.reactions:
    if i.subsystem != "":
        if i.subsystem not in tmp.keys():
            tmp[i.subsystem] = 1
        else:
            tmp[i.subsystem] += 1
tmp

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Optional, Dict, Tuple
import warnings
import os
import re
from scipy import stats
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch


def plot_publication_quality_distributions(
    batch1_data: List[str],
    batch2_data: List[str],
    reaction_ids: List[str],
    model,
    output_directory: str = ".",
    output_prefix: str = "publication_distributions",
    data_directory: str = ".",
    save_plots: bool = True,
    figsize: Tuple[int, int] = (18, 12),
    dpi: int = 300,
    x_limits: Optional[Tuple[float, float]] = None,
    debug: bool = False
) -> Dict:
   

    # Reaction information for artistic labeling
    reaction_info = {
        'HEX1': {'full_name': 'Hexokinase', 'pathway': 'Glycolysis'},
        'FBA': {'full_name': 'Fructose-bisphosphate aldolase', 'pathway': 'Glycolysis'},
        'PGK': {'full_name': 'Phosphoglycerate kinase', 'pathway': 'Glycolysis'},
        'LDH_D': {'full_name': 'Lactate dehydrogenase', 'pathway': 'Lactate metabolism'},
        'PCm': {'full_name': 'Pyruvate carboxylase', 'pathway': 'Anaplerotic reactions'},
        'CSm': {'full_name': 'Citrate synthase', 'pathway': 'TCA cycle'},
        'MDHm': {'full_name': 'Malate dehydrogenase', 'pathway': 'TCA cycle'},
        'SUCD1m': {'full_name': 'Succinate dehydrogenase', 'pathway': 'TCA cycle'},
        'EX_o2(e)': {'full_name': 'Oxygen exchange', 'pathway': 'Respiration'}
    }

    # Phase definitions: name, color, text color, inclusive timepoint range
    phase_info = {
        0: {'name': 'Early Exponential', 'color': '#E8F4F8', 'text_color': '#2C3E50', 'range': range(0, 4)},   # 0,1,2,3
        1: {'name': 'Late Exponential',  'color': '#FFF2E8', 'text_color': '#8B4513', 'range': range(4, 7)},   # 4,5,6
        2: {'name': 'Stationary',        'color': '#F0E8F8', 'text_color': '#4A148C', 'range': range(7, 11)}  # 7,8,9,10
    }

    def extract_timepoint(filename: str) -> Optional[int]:
        match = re.search(r'timepoint_(\d+)', filename)
        return int(match.group(1)) if match else None

    def load_batch_data(file_list, batch_name):
        """Load all per-timepoint sample arrays for a batch, keyed by timepoint."""
        timepoint_data = {}
        print(f"📁 Loading {batch_name} data...")

        for filename in file_list:
            tp = extract_timepoint(filename)
            if tp is None:
                continue

            filepath = Path(data_directory) / filename
            try:
                with open(filepath, 'rb') as f:
                    data = pickle.load(f)

                data_array = np.array(data)
                finite_mask = np.all(np.isfinite(data_array), axis=1)
                clean_data = data_array[finite_mask]

                if len(clean_data) >= 50:
                    timepoint_data[tp] = clean_data
                    print(f"  ✅ t{tp}: {clean_data.shape[0]} samples")
                else:
                    print(f"  ⚠️  t{tp}: only {len(clean_data)} samples - skipped")

            except Exception as e:
                print(f"  ❌ Error loading {filename}: {e}")

        return timepoint_data

    def compute_phase_averages(timepoint_data, batch_name):
        """
        For each phase, elementwise-average the sample arrays across the
        timepoints in that phase's inclusive range. 
        """
        phase_averaged = {}

        for phase_idx, info in phase_info.items():
            available_arrays = []
            available_tps = []

            for tp in info['range']:
                if tp in timepoint_data:
                    available_arrays.append(timepoint_data[tp])
                    available_tps.append(tp)

            if len(available_arrays) == 0:
                print(f"  ⚠️  {batch_name} - {info['name']}: no timepoints found in range, skipping phase")
                phase_averaged[phase_idx] = None
                continue

            # Check shape consistency before averaging elementwise
            shapes = {arr.shape for arr in available_arrays}
            if len(shapes) > 1:
                print(f"  ❌ {batch_name} - {info['name']}: mismatched sample array shapes {shapes} "
                      f"across timepoints {available_tps}. Cannot average elementwise, skipping phase.")
                phase_averaged[phase_idx] = None
                continue

            averaged = np.mean(np.stack(available_arrays, axis=0), axis=0)
            phase_averaged[phase_idx] = averaged
            print(f"  ✅ {batch_name} - {info['name']}: averaged timepoints {available_tps} "
                  f"-> shape {averaged.shape}")

        return phase_averaged

    def calculate_statistics(data1, data2):
        try:
            statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
            n1, n2 = len(data1), len(data2)
            U1 = statistic
            effect_size = 1 - (2 * U1) / (n1 * n2)

            if abs(effect_size) < 0.3:
                effect_magnitude = "Small"
            elif abs(effect_size) < 0.5:
                effect_magnitude = "Medium"
            else:
                effect_magnitude = "Large"

            return {
                'p_value': p_value,
                'effect_size': effect_size,
                'effect_magnitude': effect_magnitude,
                'n1': n1,
                'n2': n2
            }
        except Exception as e:
            print(f"Statistical calculation error: {e}")
            return None

    def format_p_value(p_val):
        if p_val < 0.001:
            return "p < 0.001***"
        elif p_val < 0.01:
            return f"p = {p_val:.3f}**"
        elif p_val < 0.05:
            return f"p = {p_val:.3f}*"
        else:
            return f"p = {p_val:.3f}"

    # Configure matplotlib for publication quality with Arial font
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
        'font.size': 30,
        'axes.titlesize': 20,
        'axes.labelsize': 20,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 16,
        'figure.titlesize': 20,
        'axes.linewidth': 1.2,
        'grid.alpha': 0.3,
        'axes.spines.top': False,
        'axes.spines.right': False,
    })

    print(f"🎨 HIGH-QUALITY PUBLICATION PLOTS (PHASE-AVERAGED, NO SCALING)")
    print(f"🎨 " + "=" * 50)

    # Get reaction indices and validate
    all_reaction_ids = [rxn.id for rxn in model.reactions]
    reaction_id_to_index = {rxn_id: idx for idx, rxn_id in enumerate(all_reaction_ids)}

    missing_reactions = [rxn for rxn in reaction_ids if rxn not in reaction_id_to_index]
    if missing_reactions:
        print(f"❌ Missing reactions: {missing_reactions}")
        return None

    # Load raw per-timepoint data
    batch1_timepoint_data = load_batch_data(batch1_data, "Batch1")
    batch2_timepoint_data = load_batch_data(batch2_data, "Batch2")

    # Compute the 3 phase-averaged datasets per batch (used directly, not saved)
    print(f"\n📊 COMPUTING PHASE AVERAGES")
    print("=" * 35)
    batch1_phase_data = compute_phase_averages(batch1_timepoint_data, "Batch1")
    batch2_phase_data = compute_phase_averages(batch2_timepoint_data, "Batch2")

    os.makedirs(output_directory, exist_ok=True)

    # Calculate subplot dimensions
    n_reactions = len(reaction_ids)
    n_phases = 3  # Always 3 phases: Early / Late / Stationary

    # Create figure with custom spacing for artistic layout
    fig = plt.figure(figsize=figsize)

    gs = fig.add_gridspec(n_reactions, n_phases,
                          left=0.15, right=0.95, top=0.85, bottom=0.1,
                          hspace=0.3, wspace=0.2)

    # Professional color scheme
    colors = {
        'batch1': '#1f77b4',      # Professional blue
        'batch2': '#d62728',      # Professional red
        'batch1_alpha': 0.7,
        'batch2_alpha': 0.7
    }

    results = {
        'reaction_ids': reaction_ids,
        'phase_definitions': {idx: {'name': info['name'], 'timepoints': list(info['range'])}
                               for idx, info in phase_info.items()},
        'statistical_results': {}
    }

    print(f"\n🧬 CREATING HIGH-QUALITY PLOTS")
    print("=" * 35)

    # Add phase headers with artistic styling
    for col_idx, info in phase_info.items():
        fig.text(0.15 + (col_idx + 0.5) * (0.8 / n_phases), 0.92, info['name'],
                 ha='center', va='center', fontsize=14, fontweight='bold',
                 color=info['text_color'],
                 bbox=dict(boxstyle="round,pad=0.3",
                          facecolor=info['color'],
                          edgecolor='none', alpha=0.8))

    # Process each reaction
    for row_idx, reaction_id in enumerate(reaction_ids):
        print(f"\n  🔬 Processing {reaction_id}...")

        reaction_idx = reaction_id_to_index[reaction_id]
        results['statistical_results'][reaction_id] = {}

        # Artistic reaction label on the left
        y_position = 0.85 - (row_idx + 0.5) * (0.75 / n_reactions)
        fig.text(0.02, y_position + 0.02, reaction_id,
                 ha='left', va='center', fontsize=16, fontweight='bold',
                 color='#2C3E50')

        # Process each phase (column)
        for col_idx, info in phase_info.items():
            batch1_phase_array = batch1_phase_data.get(col_idx)
            batch2_phase_array = batch2_phase_data.get(col_idx)

            if batch1_phase_array is None or batch2_phase_array is None:
                print(f"    ⚠️  Skipping {reaction_id} - {info['name']}: missing phase-averaged data")
                continue

            # Create subplot
            ax = fig.add_subplot(gs[row_idx, col_idx])

            # Subtle phase background
            phase_bg = Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                facecolor=info['color'],
                                alpha=0.3, zorder=0)
            ax.add_patch(phase_bg)

            # Get flux data (phase-averaged) for this reaction
            batch1_flux = batch1_phase_array[:, reaction_idx]
            batch2_flux = batch2_phase_array[:, reaction_idx]

            if debug:
                print(f"    🔍 DEBUG {reaction_id} - {info['name']}:")
                print(f"      Batch1 range: [{np.min(batch1_flux):.6e}, {np.max(batch1_flux):.6e}]")
                print(f"      Batch2 range: [{np.min(batch2_flux):.6e}, {np.max(batch2_flux):.6e}]")
                print(f"      Samples: {len(batch1_flux)}, {len(batch2_flux)}")

            # Calculate statistics
            stats_result = calculate_statistics(batch1_flux, batch2_flux)
            if stats_result:
                results['statistical_results'][reaction_id][info['name']] = stats_result

            # NO SCALING - use raw flux values
            batch1_flux_display = batch1_flux
            batch2_flux_display = batch2_flux
            xlabel = 'Flux Value'

            # Determine histogram range with explicit control
            combined_data = np.concatenate([batch1_flux_display, batch2_flux_display])

            if x_limits is not None:
                data_min, data_max = x_limits
            else:
                data_min, data_max = np.min(combined_data), np.max(combined_data)
                margin = (data_max - data_min) * 0.05 if data_max != data_min else 0.1
                data_min, data_max = data_min - margin, data_max + margin

            bins = np.linspace(data_min, data_max, 40)

            # Histograms with raw counts and explicit range
            ax.hist(batch1_flux_display, bins=bins, alpha=colors['batch1_alpha'],
                   color=colors['batch1'], density=False,
                   edgecolor='white', linewidth=0.5,
                   label='High-yielding' if row_idx == 0 and col_idx == 0 else "")
            ax.hist(batch2_flux_display, bins=bins, alpha=colors['batch2_alpha'],
                   color=colors['batch2'], density=False,
                   edgecolor='white', linewidth=0.5,
                   label='Low-yielding' if row_idx == 0 and col_idx == 0 else "")

            ax.set_xlim(data_min, data_max)

            # Median lines (raw values)
            median1 = np.median(batch1_flux_display)
            median2 = np.median(batch2_flux_display)
            ax.axvline(median1, color=colors['batch1'], linestyle='--', linewidth=2.5, alpha=0.9)
            ax.axvline(median2, color=colors['batch2'], linestyle='--', linewidth=2.5, alpha=0.9)

            # Statistical annotation
            if stats_result:
                p_text = format_p_value(stats_result['p_value'])
                effect_text = f"r = {stats_result['effect_size']:.3f}"
                stats_box = f"{p_text}\n{effect_text}"
                ax.text(0.98, 0.95, stats_box, transform=ax.transAxes,
                       fontsize=13, va='top', ha='right',
                       bbox=dict(boxstyle="round,pad=0.4",
                               facecolor='white', edgecolor='#BDC3C7',
                               alpha=0.9, linewidth=1))

            # Clean formatting
            ax.set_xlabel(xlabel, fontsize=10)
            ax.set_ylabel('Count', fontsize=10)
            ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5)
            ax.set_axisbelow(True)

            ax.ticklabel_format(style='scientific', axis='x', scilimits=(-3, 3))

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#7F8C8D')
            ax.spines['bottom'].set_color('#7F8C8D')

    # Legend at the bottom
    legend_elements = [
        mpatches.Patch(color=colors['batch1'], alpha=colors['batch1_alpha'], label='High-yielding batch'),
        mpatches.Patch(color=colors['batch2'], alpha=colors['batch2_alpha'], label='Low-yielding batch'),
        plt.Line2D([0], [0], color=colors['batch1'], linestyle='--', linewidth=2.5, alpha=0.9, label='Median (high-yielding)'),
        plt.Line2D([0], [0], color=colors['batch2'], linestyle='--', linewidth=2.5, alpha=0.9, label='Median (low-yielding)')
    ]

    fig.legend(handles=legend_elements, loc='lower center',
              bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False, fontsize=14)

    # Save in multiple high-quality formats
    if save_plots:
        base_filename = os.path.join(output_directory, f"{output_prefix}_high_quality")

        plt.savefig(f"{base_filename}.png", dpi=dpi, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        plt.savefig(f"{base_filename}.pdf", dpi=dpi, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        plt.savefig(f"{base_filename}.svg", bbox_inches='tight',
                   facecolor='white', edgecolor='none')

        print(f"\n💾 SAVED HIGH-QUALITY FILES:")
        print(f"  📄 PNG: {output_prefix}_high_quality.png")
        print(f"  📄 PDF: {output_prefix}_high_quality.pdf (vector - publication ready)")
        print(f"  📄 SVG: {output_prefix}_high_quality.svg (fully editable)")

    plt.show()

    # Save statistical results
    if save_plots:
        stats_data = []
        for reaction_id, phase_stats in results['statistical_results'].items():
            for phase_name, stat_result in phase_stats.items():
                stats_data.append({
                    'Reaction': reaction_id,
                    'Phase': phase_name,
                    'P_Value': stat_result['p_value'],
                    'Effect_Size': stat_result['effect_size'],
                    'Effect_Magnitude': stat_result['effect_magnitude'],
                    'N_Batch1': stat_result['n1'],
                    'N_Batch2': stat_result['n2']
                })

        stats_df = pd.DataFrame(stats_data)
        csv_filename = os.path.join(output_directory, f"{output_prefix}_statistics_high_quality.csv")
        stats_df.to_csv(csv_filename, index=False)
        print(f"  📊 Statistics: {os.path.basename(csv_filename)}")

    return results